# MAC 237 Graded Lab 3, Session 9: handling program input

**Wednesday, October 14, 2026. This session runs in Google Colab, in a browser.**

B124 is not available to us on Wednesday evenings, so this session is a notebook you run on
any computer with a browser: a classroom machine or your own laptop. It uses only the Python
3 standard library, needs no administrative rights, opens no socket, and writes only inside
its own folder.

**Nothing in this notebook contacts any host outside it.**

Two bugs, both in a few lines, both among the most common causes of real incidents: a name
accepted at any length, and a file name accepted as a path. Reproduce each one, then write the
version that refuses it.

## Section 1. Your personalization token

Put your own student ID in `sid` and run the cell. The token must be visible in the first and
the last screenshot of this session.

In [ ]:
import hashlib, datetime

sid = '24621670'          # put your own student ID here, in quotes
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
TOKEN = hashlib.sha256((sid + '-' + stamp).encode()).hexdigest()[:16]
print('MAC237 Lab 3 token:', TOKEN)

MAC237 Lab 3 token: 52caab241e4ad066


## Section 3. Part A. The bug

The cell below makes a `data` folder with one file in it, and one file beside the folder that
the program should never be able to read. Then it defines `buggy_read_file`, which joins the
requested name onto the folder and opens whatever that produces, and `buggy_greet`, which
accepts a name of any length.

In [ ]:
import os

os.makedirs('data', exist_ok=True)
open('data/notes.txt', 'w').write('lab notes\n')
open('secret_outside.txt', 'w').write('not for students, token ' + TOKEN + '\n')

BUGGY_BASE = 'data'

def buggy_greet(name):
    return 'Hello, ' + name + '!'                     # no length check

def buggy_read_file(path):
    return open(os.path.join(BUGGY_BASE, path)).read()  # no canonicalization

print('token:', TOKEN)
long_name = 'A' * 20000
print('greeting length accepted :', len(buggy_greet(long_name)))
leaked = buggy_read_file('../secret_outside.txt')
print('read through ../         :', leaked.strip())

assert len(buggy_greet(long_name)) > 20000
assert 'not for students' in leaked
print()
print('  CHECK PASS: the buggy version accepts any length and reads outside data')

token: 52caab241e4ad066
greeting length accepted : 20008
read through ../         : not for students, token 52caab241e4ad066

  CHECK PASS: the buggy version accepts any length and reads outside data


The second result is the whole bug: the program asked for a name and accepted a path.

## Section 3. Part B. The patch

`patched_greet` rejects a name longer than 64 characters. `patched_read_file` resolves the
requested path with `os.path.realpath` and refuses anything that does not end up inside `data`.
The first two calls below must raise, naming the reason; the third must return the notes. A
patch that also breaks the legitimate case is not a patch.

In [ ]:
BASE = os.path.realpath('data')

def patched_greet(name):
    if len(name) > 64:
        raise ValueError('name rejected: %d characters, limit is 64' % len(name))
    return 'Hello, ' + name + '!'

def patched_read_file(path):
    full = os.path.realpath(os.path.join(BASE, path))
    if full != BASE and not full.startswith(BASE + os.sep):
        raise ValueError('refusing to read outside %s: %s' % (BASE, full))
    return open(full).read()

print('token:', TOKEN)
refused = []
for label, call in (('20000-character name', lambda: patched_greet(long_name)),
                    ('../secret_outside.txt', lambda: patched_read_file('../secret_outside.txt'))):
    try:
        call()
        print(f'{label:24s} ACCEPTED')
    except ValueError as exc:
        refused.append(label)
        print(f'{label:24s} refused: {exc}')
notes = patched_read_file('notes.txt')
print(f"{'notes.txt':24s} {notes.strip()}")

assert len(refused) == 2, 'both bad inputs must be refused'
assert notes.strip() == 'lab notes', 'the legitimate read must still work'
print()
print('  CHECK PASS: the patch refuses both bad inputs and still serves the legitimate one')

token: 52caab241e4ad066
20000-character name     refused: name rejected: 20000 characters, limit is 64
../secret_outside.txt    refused: refusing to read outside /content/data: /content/secret_outside.txt
notes.txt                lab notes

  CHECK PASS: the patch refuses both bad inputs and still serves the legitimate one


## What goes in the report

Both `CHECK PASS` lines must appear above.

1. Explain why `os.path.realpath` is the operative call rather than a check for the two dots, and
   give one input that a two-dot filter would let through.
2. Say why validating input at the boundary is cheaper than defending every place the value is
   later used.
3. Cite CTPE Chapter 5 on handling program input.

**Screenshot** `lab3_s09_input_validation.png`, showing the read succeeding under the buggy version
and being refused under the patched one, with your token and your Google account name.

**Save and sign out.** Save the notebook into `lab3` with File, then Save a copy in GitHub,
under this notebook's name. Choose Runtime, then Disconnect and delete runtime, and on a shared
computer sign out of Google. Capture `lab3_s09_logout.png` while the token is still on
screen, then upload both screenshots to `lab3` on github.com with Add file, then Upload files.